# Four-Framework Showdown
## Solving One Task in LangChain, LlamaIndex, Haystack, and DSPy

**Project -- NLP / Framework Comparison Portfolio**

---

Every LLM framework promises to simplify building AI applications, but they
make very different design trade-offs. Rather than reading docs in isolation,
this notebook solves **one concrete task** in all four frameworks, then
compares developer experience and output quality side by side.

### The Task

**Question-Answering over Technical Documents with Structured Output.**

Given a small corpus of technical architecture documents, answer user
questions and return a structured JSON response containing the answer,
confidence level, and source references.

This task is small enough to implement cleanly in each framework but
touches the core abstraction of each: chains, indices, pipelines, and
modules.

## What You Will Learn

| Section | Content |
|---|---|
| 1 | Shared corpus and ground-truth answers |
| 2 | **LangChain** -- Chain + OutputParser |
| 3 | **LlamaIndex** -- Index + ResponseSynthesizer |
| 4 | **Haystack** -- Pipeline + Component |
| 5 | **DSPy** -- Module + Signature |
| 6 | Output comparison |
| 7 | Developer experience comparison |
| 8 | When to use which framework |

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Ready.')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import re, math, json as json_mod, textwrap, hashlib
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 25)
pd.set_option('display.max_colwidth', 140)
plt.rcParams['figure.figsize'] = (14, 5)
sns.set_style('whitegrid')
np.random.seed(42)
print('Imports OK')

## Part 1: Shared Corpus and Ground Truth

All four frameworks operate on the **same** corpus and answer the **same**
questions. This isolates the framework design from the data.

In [ ]:
CORPUS = [
    {'doc_id': 'ARCH-001', 'title': 'Authentication Service',
     'text': (
         'The authentication service uses JWT tokens signed with RS256. '
         'Access tokens expire after 15 minutes; refresh tokens after 7 days. '
         'Token rotation happens via a /auth/refresh endpoint. The signing '
         'keys are stored in AWS Secrets Manager and rotated monthly. '
         'Rate limiting is set to 100 requests per minute per IP for the '
         'login endpoint. Failed login attempts trigger exponential backoff '
         'after 5 consecutive failures, locking the account for 30 minutes.'
     )},
    {'doc_id': 'ARCH-002', 'title': 'Order Processing Pipeline',
     'text': (
         'Orders flow through a 4-stage pipeline: validation, payment, '
         'fulfillment, and notification. Each stage is an independent '
         'microservice connected via RabbitMQ. The validation service '
         'checks inventory availability against a Redis cache that syncs '
         'every 30 seconds from the PostgreSQL inventory table. Payment '
         'processing uses Stripe with automatic retry (3 attempts, '
         'exponential backoff). The fulfillment service integrates with '
         'the warehouse API and has a 99.5%% SLA. Notification sends '
         'order confirmation via email (SES) and SMS (Twilio).'
     )},
    {'doc_id': 'ARCH-003', 'title': 'Search Infrastructure',
     'text': (
         'Product search runs on Elasticsearch 8.x with 3 data nodes and '
         '2 coordinating nodes. Indexing is near-real-time with a 1-second '
         'refresh interval. The search pipeline applies query expansion '
         'using synonym dictionaries, spell correction via a custom '
         'analyzer, and BM25 ranking boosted by product popularity scores. '
         'Faceted search supports category, price range, brand, and rating '
         'filters. Average query latency is 45ms at the P95 level. The '
         'index is rebuilt weekly from the product catalog to handle schema '
         'changes.'
     )},
    {'doc_id': 'ARCH-004', 'title': 'Caching Strategy',
     'text': (
         'The platform uses a three-tier caching strategy. L1 is an '
         'in-process LRU cache (Caffeine) with 5-minute TTL and 10,000 '
         'entry limit. L2 is a shared Redis cluster (6 nodes, 3 primary '
         'with read replicas) using 15-minute TTL for API responses. '
         'L3 is a CDN edge cache (CloudFront) with 1-hour TTL for static '
         'assets and 5-minute TTL for personalised content. Cache '
         'invalidation uses a pub/sub channel on Redis; services subscribe '
         'to invalidation events for their namespaces. Overall cache hit '
         'rate is 94%% across all tiers.'
     )},
    {'doc_id': 'ARCH-005', 'title': 'Data Pipeline and Analytics',
     'text': (
         'Raw event data lands in S3 via Kinesis Firehose with 5-minute '
         'buffering. A Spark job on EMR runs hourly to transform and '
         'deduplicate events into Parquet format in the analytics data lake. '
         'DBT models build the analytical tables: user sessions, conversion '
         'funnels, and revenue attribution. Dashboards in Looker refresh '
         'every 15 minutes from a Snowflake warehouse that mirrors the data '
         'lake. Data retention is 2 years for aggregated data and 90 days '
         'for raw events. PII is masked using a deterministic tokeniser '
         'before entering the lake.'
     )},
    {'doc_id': 'ARCH-006', 'title': 'Deployment and CI/CD',
     'text': (
         'All services deploy via GitHub Actions. The pipeline runs linting '
         '(ruff), type checking (mypy), unit tests (pytest, 80%% coverage '
         'gate), integration tests against a Docker Compose environment, '
         'and security scanning (Snyk). Deployments use blue-green strategy '
         'on ECS Fargate with automatic rollback if the health check fails '
         'within 5 minutes. Feature flags are managed via LaunchDarkly. '
         'Canary releases target 5%% of traffic for 30 minutes before full '
         'rollout. Rollback SLA is under 3 minutes.'
     )},
    {'doc_id': 'ARCH-007', 'title': 'Monitoring and Alerting',
     'text': (
         'Observability uses the three pillars: metrics (Prometheus + Grafana), '
         'logs (Fluentd to Elasticsearch + Kibana), and traces (OpenTelemetry '
         'to Jaeger). Alerting rules in Prometheus fire PagerDuty incidents '
         'for P1/P2 severities and Slack notifications for P3/P4. SLO '
         'dashboards track four golden signals: latency, traffic, errors, '
         'and saturation. Error budget burn rate alerts trigger when the '
         'monthly error budget is consumed faster than 2x the normal rate. '
         'On-call rotation is weekly with a primary and secondary responder.'
     )},
    {'doc_id': 'ARCH-008', 'title': 'API Gateway and Rate Limiting',
     'text': (
         'Kong API Gateway handles routing, authentication verification, '
         'and rate limiting. Rate limits use a sliding window algorithm: '
         '1,000 requests per minute for authenticated users, 60 per minute '
         'for anonymous users. API versioning follows URL path strategy '
         '(/v1/, /v2/). Request/response transformation plugins normalise '
         'headers and inject correlation IDs. The gateway runs on 4 pods '
         'with horizontal pod autoscaling triggered at 70%% CPU. Average '
         'gateway overhead is 3ms per request.'
     )},
]

print(f'Corpus: {len(CORPUS)} architecture documents')
for d in CORPUS:
    print(f"  {d['doc_id']}: {d['title']} ({len(d['text'].split())} words)")

In [ ]:
# Evaluation queries with ground truth
EVAL_QUERIES = [
    {
        'query': 'How long do access tokens last and what algorithm signs them?',
        'expected_doc': 'ARCH-001',
        'expected_fields': {
            'answer': 'Access tokens expire after 15 minutes, signed with RS256.',
            'confidence': 'high',
            'source_doc': 'ARCH-001',
        },
    },
    {
        'query': 'What message broker connects the order processing stages?',
        'expected_doc': 'ARCH-002',
        'expected_fields': {
            'answer': 'RabbitMQ connects the four order processing stages.',
            'confidence': 'high',
            'source_doc': 'ARCH-002',
        },
    },
    {
        'query': 'What is the P95 search latency and how many Elasticsearch nodes run?',
        'expected_doc': 'ARCH-003',
        'expected_fields': {
            'answer': 'P95 latency is 45ms on a 5-node cluster (3 data + 2 coordinating).',
            'confidence': 'high',
            'source_doc': 'ARCH-003',
        },
    },
    {
        'query': 'What is the overall cache hit rate and what are the three cache tiers?',
        'expected_doc': 'ARCH-004',
        'expected_fields': {
            'answer': '94% hit rate across L1 (in-process LRU), L2 (Redis), and L3 (CDN).',
            'confidence': 'high',
            'source_doc': 'ARCH-004',
        },
    },
    {
        'query': 'How is PII handled in the data pipeline?',
        'expected_doc': 'ARCH-005',
        'expected_fields': {
            'answer': 'PII is masked using a deterministic tokeniser before entering the data lake.',
            'confidence': 'high',
            'source_doc': 'ARCH-005',
        },
    },
    {
        'query': 'What deployment strategy is used and how fast can a rollback happen?',
        'expected_doc': 'ARCH-006',
        'expected_fields': {
            'answer': 'Blue-green deployment on ECS Fargate with rollback SLA under 3 minutes.',
            'confidence': 'high',
            'source_doc': 'ARCH-006',
        },
    },
    {
        'query': 'What are the four golden signals tracked in the SLO dashboards?',
        'expected_doc': 'ARCH-007',
        'expected_fields': {
            'answer': 'Latency, traffic, errors, and saturation.',
            'confidence': 'high',
            'source_doc': 'ARCH-007',
        },
    },
    {
        'query': 'What rate limits apply to anonymous API users?',
        'expected_doc': 'ARCH-008',
        'expected_fields': {
            'answer': '60 requests per minute for anonymous users via sliding window.',
            'confidence': 'high',
            'source_doc': 'ARCH-008',
        },
    },
]

print(f'{len(EVAL_QUERIES)} evaluation queries')
for i, q in enumerate(EVAL_QUERIES, 1):
    print(f"  Q{i}: {q['query'][:65]}..." if len(q['query']) > 65 else f"  Q{i}: {q['query']}")

## Shared Internals

All four framework implementations share the same retriever and LLM
simulation layer. The frameworks differ only in how they **compose** these
building blocks.

In [ ]:
class MiniEmbedder:
    """Tiny TF-IDF-style embedder for retrieval."""
    def __init__(self, dim=64):
        self.dim, self.vocab, self.idf, self.proj = dim, {}, {}, None

    def fit(self, texts):
        df = Counter()
        for t in texts:
            for tok in set(re.findall(r'[a-z0-9]+', t.lower())):
                df[tok] += 1
        self.vocab = {t: i for i, t in enumerate(sorted(df))}
        n = len(texts)
        self.idf = {t: math.log(n / (1 + c)) for t, c in df.items()}
        np.random.seed(42)
        self.proj = np.random.randn(len(self.vocab), self.dim) / math.sqrt(self.dim)

    def encode(self, text):
        vec = np.zeros(len(self.vocab))
        for tok, cnt in Counter(re.findall(r'[a-z0-9]+', text.lower())).items():
            if tok in self.vocab:
                vec[self.vocab[tok]] = cnt * self.idf.get(tok, 1.0)
        p = vec @ self.proj
        norm = np.linalg.norm(p)
        return p / norm if norm > 0 else p


embedder = MiniEmbedder(64)
embedder.fit([d['text'] for d in CORPUS])
DOC_EMBEDDINGS = {d['doc_id']: embedder.encode(d['text']) for d in CORPUS}
DOC_MAP = {d['doc_id']: d for d in CORPUS}


def retrieve_docs(query, top_k=3):
    """Shared retriever -- same for all frameworks."""
    qe = embedder.encode(query)
    scores = [(did, float(emb @ qe)) for did, emb in DOC_EMBEDDINGS.items()]
    scores.sort(key=lambda x: -x[1])
    return [DOC_MAP[did] for did, _ in scores[:top_k]]


def simulate_llm(prompt, doc_text, query):
    """Deterministic LLM simulation.

    Extracts the most relevant sentence from the document,
    then returns a structured response.
    """
    sentences = re.split(r'(?<=[.!?])\s+', doc_text)
    q_tokens = set(re.findall(r'[a-z0-9]+', query.lower()))
    scored = []
    for s in sentences:
        s_tokens = set(re.findall(r'[a-z0-9]+', s.lower()))
        overlap = len(q_tokens & s_tokens)
        scored.append((overlap, s))
    scored.sort(key=lambda x: -x[0])
    # Take top-2 sentences as the answer
    best = scored[:2]
    answer = ' '.join(s for _, s in best)
    confidence = 'high' if best[0][0] >= 3 else 'medium' if best[0][0] >= 2 else 'low'
    return answer.strip(), confidence


# Test shared components
test_docs = retrieve_docs('access token expiry')
print(f'Top retrieval for "access token expiry": {test_docs[0]["doc_id"]} -- {test_docs[0]["title"]}')
ans, conf = simulate_llm('', test_docs[0]['text'], 'access token expiry')
print(f'LLM response: {ans[:100]}... (confidence: {conf})')

## Part 2: LangChain Implementation

### Design Philosophy

LangChain composes applications as **chains** -- sequences of callables
connected with the `|` (pipe) operator via LCEL (LangChain Expression Language).

```
                   ┌─────────────┐
            query  │  Retriever  │  docs
          ────────>│  .invoke()  │────────┐
                   └─────────────┘        │
                                          ▼
                   ┌─────────────┐  ┌──────────────┐  ┌──────────────┐
                   │   Prompt    │──│    ChatLLM    │──│ OutputParser  │
                   │  Template   │  │  .invoke()    │  │ (JSON/Pydant)│
                   └─────────────┘  └──────────────┘  └──────────────┘
```

**Key concepts:** LCEL chains, Runnables, PromptTemplate, OutputParser, Pydantic models.

In [ ]:
class LangChainRetriever:
    """Simulates langchain_community VectorStoreRetriever."""
    def invoke(self, query, top_k=3):
        return retrieve_docs(query, top_k)


class LangChainPromptTemplate:
    """Simulates langchain_core.prompts.ChatPromptTemplate."""
    def __init__(self, template):
        self.template = template

    def format(self, **kwargs):
        result = self.template
        for k, v in kwargs.items():
            result = result.replace('{' + k + '}', str(v))
        return result


class LangChainJsonOutputParser:
    """Simulates langchain_core.output_parsers.JsonOutputParser."""
    def parse(self, text, doc_id):
        return {'answer': text[0], 'confidence': text[1], 'source_doc': doc_id}


class LangChainQAChain:
    """Simulates a full LangChain LCEL chain:
       retriever | prompt | llm | output_parser
    """
    def __init__(self):
        self.retriever = LangChainRetriever()
        self.prompt = LangChainPromptTemplate(
            'Given this context: {context}\n'
            'Answer the question: {query}\n'
            'Return JSON with: answer, confidence, source_doc'
        )
        self.parser = LangChainJsonOutputParser()

    def invoke(self, query):
        docs = self.retriever.invoke(query, top_k=3)
        best_doc = docs[0]
        prompt_text = self.prompt.format(
            context=best_doc['text'], query=query
        )
        answer, confidence = simulate_llm(prompt_text, best_doc['text'], query)
        return self.parser.parse((answer, confidence), best_doc['doc_id'])


langchain_chain = LangChainQAChain()
lc_result = langchain_chain.invoke('How long do access tokens last?')
print('LangChain output:')
print(json_mod.dumps(lc_result, indent=2))

### LangChain -- Real Code

```python
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from pydantic import BaseModel

class StructuredAnswer(BaseModel):
    answer: str
    confidence: str
    source_doc: str

embeddings = HuggingFaceEmbeddings(model_name='BAAI/bge-small-en-v1.5')
vectorstore = Chroma.from_texts([d['text'] for d in corpus], embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Answer from the context. Return JSON: answer, confidence, source_doc.\n'
               '{format_instructions}'),
    ('human', 'Context: {context}\nQuestion: {query}'),
])
parser = JsonOutputParser(pydantic_object=StructuredAnswer)
llm = ChatOllama(model='qwen3')

# LCEL chain
chain = prompt | llm | parser

docs = retriever.invoke(query)
result = chain.invoke({
    'context': docs[0].page_content,
    'query': query,
    'format_instructions': parser.get_format_instructions()
})
```

**Lines of code:** ~30 | **Key abstraction:** LCEL chain (`|` pipes)

## Part 3: LlamaIndex Implementation

### Design Philosophy

LlamaIndex centres on **indices** -- data structures that organize your
documents for efficient retrieval. The query engine combines retrieval
and synthesis in a single abstraction.

```
                   ┌──────────────┐
            docs   │  VectorStore │
          ────────>│    Index     │
                   └──────────────┘
                         │
                         ▼
                   ┌──────────────┐
            query  │ QueryEngine  │  structured response
          ────────>│ .query()     │──────────────────────>
                   └──────────────┘
                     (retriever +
                      response synth
                      + output parser)
```

**Key concepts:** VectorStoreIndex, QueryEngine, ResponseSynthesizer, Pydantic program.

In [ ]:
class LlamaIndexVectorStore:
    """Simulates llama_index VectorStoreIndex."""
    def __init__(self, documents):
        self.documents = documents

    def as_retriever(self, top_k=3):
        return LlamaIndexRetriever(self.documents, top_k)


class LlamaIndexRetriever:
    def __init__(self, documents, top_k):
        self.documents = documents
        self.top_k = top_k

    def retrieve(self, query):
        return retrieve_docs(query, self.top_k)


class LlamaIndexResponseSynthesizer:
    """Simulates ResponseSynthesizer with structured output."""
    def synthesize(self, query, docs):
        best = docs[0]
        answer, confidence = simulate_llm('', best['text'], query)
        return {
            'answer': answer,
            'confidence': confidence,
            'source_doc': best['doc_id'],
            'source_nodes': [d['doc_id'] for d in docs],
        }


class LlamaIndexQueryEngine:
    """Simulates llama_index QueryEngine.
    High-level abstraction: retrieval + synthesis in one call.
    """
    def __init__(self, index):
        self.retriever = index.as_retriever(top_k=3)
        self.synthesizer = LlamaIndexResponseSynthesizer()

    def query(self, query_str):
        nodes = self.retriever.retrieve(query_str)
        return self.synthesizer.synthesize(query_str, nodes)


li_index = LlamaIndexVectorStore(CORPUS)
li_engine = LlamaIndexQueryEngine(li_index)
li_result = li_engine.query('How long do access tokens last?')
print('LlamaIndex output:')
print(json_mod.dumps(li_result, indent=2))

### LlamaIndex -- Real Code

```python
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.output_parsers import PydanticOutputParser
from pydantic import BaseModel

class StructuredAnswer(BaseModel):
    answer: str
    confidence: str
    source_doc: str

Settings.llm = Ollama(model='qwen3')
Settings.embed_model = HuggingFaceEmbedding('BAAI/bge-small-en-v1.5')

documents = [Document(text=d['text'], metadata={'doc_id': d['doc_id']})
             for d in corpus]
index = VectorStoreIndex.from_documents(documents)

query_engine = index.as_query_engine(
    output_parser=PydanticOutputParser(StructuredAnswer),
    similarity_top_k=3,
)
result = query_engine.query('How long do access tokens last?')
```

**Lines of code:** ~20 | **Key abstraction:** QueryEngine (retrieval + synthesis combined)

## Part 4: Haystack Implementation

### Design Philosophy

Haystack uses **components** connected in a **pipeline** DAG.
Each component declares its input/output types. The pipeline validates
connections at build time.

```
  ┌────────────┐     ┌───────────────┐     ┌──────────┐     ┌───────────┐
  │ DocStore + │     │ PromptBuilder │     │ Generator│     │ Output    │
  │ Retriever  │────>│               │────>│ (LLM)    │────>│ Adapter   │
  └────────────┘     └───────────────┘     └──────────┘     └───────────┘
      docs                prompt              text              JSON
```

**Key concepts:** Pipeline (DAG), Component protocol, DocumentStore, PromptBuilder.

In [ ]:
class HaystackDocumentStore:
    """Simulates haystack InMemoryDocumentStore."""
    def __init__(self):
        self.documents = []

    def write_documents(self, docs):
        self.documents = docs


class HaystackRetriever:
    """Simulates haystack InMemoryBM25Retriever."""
    def __init__(self, doc_store):
        self.store = doc_store

    def run(self, query, top_k=3):
        return {'documents': retrieve_docs(query, top_k)}


class HaystackPromptBuilder:
    """Simulates haystack PromptBuilder."""
    def __init__(self, template):
        self.template = template

    def run(self, documents, query):
        ctx = documents[0]['text'] if documents else ''
        prompt = self.template.replace('{{ context }}', ctx).replace('{{ query }}', query)
        return {'prompt': prompt}


class HaystackGenerator:
    """Simulates haystack OllamaGenerator."""
    def run(self, prompt, documents, query):
        best = documents[0]
        answer, confidence = simulate_llm(prompt, best['text'], query)
        return {'replies': [json_mod.dumps({
            'answer': answer, 'confidence': confidence,
            'source_doc': best['doc_id'],
        })]}


class HaystackOutputAdapter:
    """Simulates haystack OutputAdapter -- parses JSON."""
    def run(self, replies):
        return {'output': json_mod.loads(replies[0])}


class HaystackPipeline:
    """Simulates haystack Pipeline with explicit wiring."""
    def __init__(self):
        self.store = HaystackDocumentStore()
        self.store.write_documents(CORPUS)
        self.retriever = HaystackRetriever(self.store)
        self.prompt_builder = HaystackPromptBuilder(
            'Context: {{ context }}\nQuestion: {{ query }}\n'
            'Return JSON: answer, confidence, source_doc'
        )
        self.generator = HaystackGenerator()
        self.output_adapter = HaystackOutputAdapter()

    def run(self, query):
        # Haystack executes components in topological order
        r1 = self.retriever.run(query, top_k=3)
        r2 = self.prompt_builder.run(r1['documents'], query)
        r3 = self.generator.run(r2['prompt'], r1['documents'], query)
        r4 = self.output_adapter.run(r3['replies'])
        return r4['output']


hs_pipe = HaystackPipeline()
hs_result = hs_pipe.run('How long do access tokens last?')
print('Haystack output:')
print(json_mod.dumps(hs_result, indent=2))

### Haystack -- Real Code

```python
from haystack import Pipeline
from haystack.components.retrievers import InMemoryBM25Retriever
from haystack.components.builders import PromptBuilder
from haystack.components.generators import OllamaGenerator
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack import Document

store = InMemoryDocumentStore()
store.write_documents([Document(content=d['text'], meta={'doc_id': d['doc_id']})
                       for d in corpus])

template = '''
Context: {% for doc in documents %}{{ doc.content }}{% endfor %}
Question: {{ query }}
Return JSON with: answer, confidence, source_doc
'''

pipe = Pipeline()
pipe.add_component('retriever', InMemoryBM25Retriever(store))
pipe.add_component('prompt', PromptBuilder(template=template))
pipe.add_component('llm', OllamaGenerator(model='qwen3'))
pipe.connect('retriever.documents', 'prompt.documents')
pipe.connect('prompt.prompt', 'llm.prompt')

result = pipe.run({'retriever': {'query': q}, 'prompt': {'query': q}})
```

**Lines of code:** ~25 | **Key abstraction:** Pipeline DAG with typed connections

## Part 5: DSPy Implementation

### Design Philosophy

DSPy treats prompting as a **programming** problem, not a writing problem.
You declare **what** the LLM should do (via Signatures), not **how** to
prompt it. The optimiser finds the best prompts automatically.

```
  ┌─────────────────────────┐
  │  Signature (declarative)│    'context, question -> answer, confidence'
  └─────────────────────────┘
              │
              ▼
  ┌─────────────────────────┐
  │  Module (program)       │    RAGModule(dspy.Module):
  │   retrieve + predict    │        retrieve -> ChainOfThought(signature)
  └─────────────────────────┘
              │
              ▼
  ┌─────────────────────────┐
  │  Optimiser (automatic)  │    BootstrapFewShot selects best demos
  └─────────────────────────┘
```

**Key concepts:** Signature, Module, Predict/ChainOfThought, Optimiser.

In [ ]:
class DSPySignature:
    """Simulates dspy.Signature -- declares I/O fields."""
    def __init__(self, sig_string):
        parts = sig_string.split('->')
        self.inputs = [f.strip() for f in parts[0].split(',')]
        self.outputs = [f.strip() for f in parts[1].split(',')]

    def __repr__(self):
        ins = ', '.join(self.inputs)
        outs = ', '.join(self.outputs)
        return f'Signature({ins} -> {outs})'


class DSPyChainOfThought:
    """Simulates dspy.ChainOfThought -- adds reasoning before answer."""
    def __init__(self, signature):
        self.sig = signature if isinstance(signature, DSPySignature) \
            else DSPySignature(signature)

    def __call__(self, context, question):
        answer, confidence = simulate_llm('', context, question)
        reasoning = f'The context mentions relevant information. Extracting key facts about: {question[:40]}'
        return type('Prediction', (), {
            'answer': answer, 'confidence': confidence,
            'reasoning': reasoning,
        })()


class DSPyRetrieve:
    """Simulates dspy.Retrieve."""
    def __init__(self, k=3):
        self.k = k

    def __call__(self, query):
        return retrieve_docs(query, self.k)


class DSPyRAGModule:
    """Simulates a DSPy Module.

    In DSPy, you declare components in __init__
    and wire them in forward().
    """
    def __init__(self):
        self.retrieve = DSPyRetrieve(k=3)
        self.generate = DSPyChainOfThought(
            'context, question -> answer, confidence'
        )

    def forward(self, question):
        docs = self.retrieve(question)
        context = docs[0]['text']
        pred = self.generate(context=context, question=question)
        return {
            'answer': pred.answer,
            'confidence': pred.confidence,
            'reasoning': pred.reasoning,
            'source_doc': docs[0]['doc_id'],
        }


dspy_module = DSPyRAGModule()
dspy_result = dspy_module.forward('How long do access tokens last?')
print('DSPy output:')
print(json_mod.dumps(dspy_result, indent=2))

### DSPy -- Real Code

```python
import dspy
from dspy import ChainOfThought, Retrieve, Module

lm = dspy.LM('ollama_chat/qwen3')
dspy.configure(lm=lm)

class QASignature(dspy.Signature):
    """Answer from architecture docs with confidence."""
    context: str = dspy.InputField(desc='Retrieved doc text')
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()
    confidence: str = dspy.OutputField(desc='high, medium, or low')

class RAGModule(dspy.Module):
    def __init__(self):
        self.retrieve = Retrieve(k=3)
        self.generate = ChainOfThought(QASignature)

    def forward(self, question):
        docs = self.retrieve(question).passages
        return self.generate(context=docs[0], question=question)

module = RAGModule()
result = module(question='How long do access tokens last?')
# result.answer, result.confidence
```

**Lines of code:** ~22 | **Key abstraction:** Signature + Module (prompt is compiled, not written)

## Part 6: Run All Frameworks on All Queries

Every framework answers the same 8 questions. We collect outputs
and measure quality.

In [ ]:
def run_all_frameworks(queries):
    lc_chain = LangChainQAChain()
    li_engine = LlamaIndexQueryEngine(LlamaIndexVectorStore(CORPUS))
    hs_pipe = HaystackPipeline()
    dspy_mod = DSPyRAGModule()

    frameworks = {
        'LangChain': lambda q: lc_chain.invoke(q),
        'LlamaIndex': lambda q: li_engine.query(q),
        'Haystack': lambda q: hs_pipe.run(q),
        'DSPy': lambda q: dspy_mod.forward(q),
    }

    rows = []
    for q in queries:
        for fw_name, fn in frameworks.items():
            result = fn(q['query'])
            # Check correctness
            correct_doc = result.get('source_doc', '') == q['expected_doc']
            gt_answer = q['expected_fields']['answer'].lower()
            pred_answer = result.get('answer', '').lower()
            # Keyword overlap as answer quality proxy
            gt_kw = set(re.findall(r'[a-z0-9]+', gt_answer)) - {'the','a','is','and','with'}
            pred_kw = set(re.findall(r'[a-z0-9]+', pred_answer)) - {'the','a','is','and','with'}
            overlap = len(gt_kw & pred_kw) / len(gt_kw) if gt_kw else 0

            rows.append({
                'framework': fw_name,
                'query': q['query'][:55],
                'correct_retrieval': correct_doc,
                'answer_overlap': round(overlap, 3),
                'confidence': result.get('confidence', 'unknown'),
                'answer_preview': result.get('answer', '')[:80],
            })

    return pd.DataFrame(rows)


results_df = run_all_frameworks(EVAL_QUERIES)
print(f'Results: {len(results_df)} rows ({len(EVAL_QUERIES)} queries x 4 frameworks)')
results_df.head(8)

In [ ]:
# Framework summary
fw_summary = results_df.groupby('framework').agg(
    retrieval_acc=('correct_retrieval', 'mean'),
    avg_answer_overlap=('answer_overlap', 'mean'),
).round(4)
fw_summary['composite'] = ((fw_summary['retrieval_acc'] + fw_summary['avg_answer_overlap']) / 2).round(4)
fw_summary = fw_summary.sort_values('composite', ascending=False)

print('FRAMEWORK OUTPUT QUALITY')
print('=' * 60)
print(fw_summary.to_string())
print()
print('Note: All frameworks use the same retriever and LLM simulation,\n'
      'so output quality is similar. The real differentiation is in\n'
      'developer experience (next section).')

## Part 7: Developer Experience Comparison

Output quality is similar because all four frameworks call the same LLM.
The real difference is **how you build, debug, and maintain** the pipeline.

In [ ]:
DX_METRICS = {
    'LangChain': {
        'loc_real': 30,
        'abstractions': 5,  # Retriever, PromptTemplate, ChatModel, OutputParser, Chain
        'prompt_control': 9,  # Full prompt template control
        'composability': 9,  # LCEL pipes, runnables
        'debuggability': 6,  # Chain internals can be opaque
        'structured_output': 8,  # JsonOutputParser, Pydantic
        'learning_curve': 7,  # Many concepts, good docs
        'ecosystem_size': 10,  # Largest ecosystem
        'optimisability': 4,  # Manual prompt tuning
    },
    'LlamaIndex': {
        'loc_real': 20,
        'abstractions': 3,  # Index, QueryEngine, Settings
        'prompt_control': 6,  # Defaults handle most; customisation is deeper
        'composability': 7,  # Query pipelines, sub-question engine
        'debuggability': 7,  # Good observability, node details
        'structured_output': 8,  # Pydantic program
        'learning_curve': 8,  # Fewer concepts, fast start
        'ecosystem_size': 8,  # Strong data connectors
        'optimisability': 5,  # Some tuning knobs
    },
    'Haystack': {
        'loc_real': 25,
        'abstractions': 4,  # DocumentStore, Retriever, Builder, Generator
        'prompt_control': 8,  # Jinja templates, PromptBuilder
        'composability': 8,  # Pipeline DAG, typed I/O
        'debuggability': 9,  # Pipeline graph, component isolation
        'structured_output': 7,  # OutputAdapter, custom components
        'learning_curve': 7,  # Clear component model
        'ecosystem_size': 6,  # Smaller but focused
        'optimisability': 5,  # Component-level tuning
    },
    'DSPy': {
        'loc_real': 22,
        'abstractions': 3,  # Signature, Module, Optimiser
        'prompt_control': 3,  # Intentionally abstracted away
        'composability': 7,  # Module nesting
        'debuggability': 6,  # Compiled prompts can be inspected
        'structured_output': 7,  # TypedPredictor
        'learning_curve': 6,  # Novel paradigm
        'ecosystem_size': 5,  # Growing but smaller
        'optimisability': 10,  # Core design goal
    },
}

dx_df = pd.DataFrame(DX_METRICS).T
dx_df.index.name = 'framework'
print('DEVELOPER EXPERIENCE SCORECARD (1-10 scale)')
print('=' * 80)
print(dx_df.to_string())

## Visualisations

In [ ]:
FW_COLORS = {
    'LangChain': '#2ecc71',
    'LlamaIndex': '#3498db',
    'Haystack': '#e67e22',
    'DSPy': '#9b59b6',
}

# 1  Radar -- DX dimensions
dims = ['prompt_control', 'composability', 'debuggability',
        'structured_output', 'learning_curve', 'ecosystem_size', 'optimisability']

fig = go.Figure()
for fw in ['LangChain', 'LlamaIndex', 'Haystack', 'DSPy']:
    vals = [DX_METRICS[fw][d] for d in dims]
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]], theta=dims + [dims[0]],
        name=fw, fill='toself',
        line_color=FW_COLORS[fw], opacity=0.5,
    ))
fig.update_layout(
    polar=dict(radialaxis=dict(range=[0, 10], visible=True, tickvals=[2,4,6,8,10])),
    title='Developer Experience Radar',
    template='plotly_white', height=520,
)
fig.show()

In [ ]:
# 2  Lines of code comparison
loc_data = [{'framework': fw, 'lines_of_code': m['loc_real'],
             'abstractions': m['abstractions']}
            for fw, m in DX_METRICS.items()]
loc_df = pd.DataFrame(loc_data).sort_values('lines_of_code')

fig = make_subplots(rows=1, cols=2, subplot_titles=['Lines of Code (Real Implementation)',
                                                     'Number of Abstractions'])
fig.add_trace(go.Bar(
    x=loc_df['framework'], y=loc_df['lines_of_code'],
    marker_color=[FW_COLORS[fw] for fw in loc_df['framework']],
    text=loc_df['lines_of_code'], textposition='outside',
), row=1, col=1)
fig.add_trace(go.Bar(
    x=loc_df['framework'], y=loc_df['abstractions'],
    marker_color=[FW_COLORS[fw] for fw in loc_df['framework']],
    text=loc_df['abstractions'], textposition='outside',
), row=1, col=2)
fig.update_layout(template='plotly_white', height=400, showlegend=False)
fig.show()

In [ ]:
# 3  Output quality heatmap (retrieval accuracy by query x framework)
hm_df = results_df.pivot_table(
    index='query', columns='framework',
    values='correct_retrieval', aggfunc='mean'
)
hm_df = hm_df[['LangChain', 'LlamaIndex', 'Haystack', 'DSPy']]

fig = px.imshow(
    hm_df.values, x=list(hm_df.columns), y=list(hm_df.index),
    color_continuous_scale='YlGn', text_auto='.0f',
    title='Retrieval Accuracy by Query and Framework',
    labels=dict(x='Framework', y='Query', color='Correct'),
)
fig.update_layout(template='plotly_white', height=480)
fig.show()

In [ ]:
# 4  Answer overlap distribution
fig = px.box(
    results_df, x='framework', y='answer_overlap',
    color='framework', color_discrete_map=FW_COLORS,
    title='Answer Quality Distribution (Keyword Overlap with Ground Truth)',
    labels={'answer_overlap': 'Answer Overlap', 'framework': 'Framework'},
    template='plotly_white', height=420,
)
fig.show()

In [ ]:
# 5  Strength/Weakness summary
strengths = {
    'LangChain': {'best': 'Ecosystem & Composability',
                  'best_dim': 'ecosystem_size',
                  'weakness': 'Prompt Optimisation',
                  'weak_dim': 'optimisability'},
    'LlamaIndex': {'best': 'Learning Curve & Speed to MVP',
                   'best_dim': 'learning_curve',
                   'weakness': 'Prompt Control',
                   'weak_dim': 'prompt_control'},
    'Haystack': {'best': 'Debuggability & Type Safety',
                 'best_dim': 'debuggability',
                 'weakness': 'Ecosystem Size',
                 'weak_dim': 'ecosystem_size'},
    'DSPy': {'best': 'Automatic Optimisation',
             'best_dim': 'optimisability',
             'weakness': 'Prompt Control',
             'weak_dim': 'prompt_control'},
}

fig = go.Figure()
fws = list(strengths.keys())
for i, fw in enumerate(fws):
    s = strengths[fw]
    fig.add_trace(go.Bar(
        x=[fw], y=[DX_METRICS[fw][s['best_dim']]],
        name=f"Strength: {s['best']}", marker_color=FW_COLORS[fw],
        text=[f"Strong: {s['best']}"], textposition='inside',
        showlegend=False,
    ))
    fig.add_trace(go.Bar(
        x=[fw], y=[-DX_METRICS[fw][s['weak_dim']]],
        name=f"Weakness: {s['weakness']}",
        marker_color=FW_COLORS[fw], opacity=0.4,
        text=[f"Weak: {s['weakness']}"], textposition='inside',
        showlegend=False,
    ))

fig.update_layout(
    title='Framework Strengths vs Weaknesses',
    barmode='relative', template='plotly_white', height=420,
    yaxis_title='Score (positive=strength, negative=weakness)',
    yaxis_range=[-10, 10],
)
fig.show()

In [ ]:
# 6  Composite DX score
dx_cols = ['prompt_control', 'composability', 'debuggability',
           'structured_output', 'learning_curve', 'ecosystem_size', 'optimisability']
dx_scores = {fw: np.mean([DX_METRICS[fw][c] for c in dx_cols]) for fw in DX_METRICS}
dx_rank = sorted(dx_scores.items(), key=lambda x: -x[1])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(len(dx_rank))), y=[v for _, v in dx_rank],
    mode='lines', line=dict(color='#bdc3c7', width=2, dash='dot'), showlegend=False,
))
for i, (fw, score) in enumerate(dx_rank):
    fig.add_trace(go.Scatter(
        x=[i], y=[score], mode='markers+text',
        marker=dict(size=22, color=FW_COLORS[fw],
                    line=dict(color='white', width=2)),
        text=[f'{score:.1f}'], textposition='top center',
        name=fw, showlegend=True,
    ))
fig.update_layout(
    title='Composite DX Score (Average Across All Dimensions)',
    xaxis=dict(tickmode='array', tickvals=list(range(len(dx_rank))),
               ticktext=[fw for fw, _ in dx_rank]),
    yaxis_title='Score', yaxis_range=[0, 10],
    template='plotly_white', height=420,
)
fig.show()

## Part 8: When to Use Which Framework

| Scenario | Recommended | Why |
|---|---|---|
| **Prototype RAG quickly** | LlamaIndex | Fewest lines, index-centric, fast start |
| **Complex multi-step chains** | LangChain | Best composability, largest ecosystem |
| **Production data pipeline** | Haystack | Typed components, pipeline DAG, debuggable |
| **Optimise prompt quality** | DSPy | Automatic prompt compilation, no prompt writing |
| **Enterprise with many integrations** | LangChain | 700+ integrations via langchain-community |
| **Structured data + text hybrid** | LlamaIndex | Native PandasQueryEngine, SQL engine |
| **Need reproducible pipelines** | Haystack | Serialisable YAML pipelines, type checks |
| **Research / academic NLP** | DSPy | Focus on optimisation metrics, not prompts |

### The Deeper Trade-Off

| | **Control** | **Convenience** | **Optimisation** |
|---|---|---|---|
| LangChain | High (LCEL) | Medium | Manual |
| LlamaIndex | Medium | High (defaults) | Manual |
| Haystack | High (components) | Medium | Manual |
| DSPy | Low (intentional) | Medium | **Automatic** |

LangChain, LlamaIndex, and Haystack give you control over **what prompt the
LLM sees**. DSPy deliberately takes that control away and optimises it
programmatically. This is the fundamental philosophical divide.

### Framework Evolution (2023--2026)

| Framework | Origin | v1 Focus | Current Direction |
|---|---|---|---|
| LangChain | 2022 | Chain of LLM calls | LCEL, LangGraph (agents), LangSmith (observability) |
| LlamaIndex | 2022 | Document indexing | Knowledge graphs, multi-modal, agent workflows |
| Haystack | 2019 (deepset) | NLP pipelines | LLM components, pipeline serialisation, model serving |
| DSPy | 2023 (Stanford) | Prompt optimisation | Typed predictors, assertions, multi-model | 

## Summary

### What We Built

One task -- **question answering with structured JSON output** -- implemented
in four frameworks:

| Framework | Abstraction | Killer Feature | Watch Out |
|---|---|---|---|
| **LangChain** | LCEL chains | Ecosystem, composability | Abstraction overhead |
| **LlamaIndex** | Index + QueryEngine | Speed to prototype | Less control |
| **Haystack** | Pipeline DAG | Type safety, debuggability | Smaller community |
| **DSPy** | Signature + Module | Automatic optimisation | No prompt control |

### Key Takeaways

| # | Insight |
|---|---|
| 1 | **Output quality is similar** -- all frameworks call the same LLMs |
| 2 | **Developer experience differs** -- composition style, debugging, learning curve |
| 3 | **No single best framework** -- the right choice depends on your context |
| 4 | **Prototyping favours LlamaIndex** -- fewest lines, highest convenience |
| 5 | **Production favours Haystack** -- typed pipelines, component isolation |
| 6 | **Scale favours LangChain** -- largest ecosystem, most integrations |
| 7 | **Quality favours DSPy** -- automatic prompt optimisation for measurable gains |

---
*Educational notebook -- NLP / Framework Comparison Portfolio.*